# Coleta de Dados Climáticos (INMET) e Saneamento (IBGE)


In [ ]:
import requests
import zipfile
import os
import pandas as pd
from io import BytesIO
import glob

# Download dos ZIPs anuais (dados de estações automáticas)
# URL padrão: https://portal.inmet.gov.br/uploads/dadoshistoricos/{ANO}.zip
years = range(2010, 2024)

for year in years:
    url = f"https://portal.inmet.gov.br/uploads/dadoshistoricos/{year}.zip"
    print(f"Baixando {year}...")

    try:
        resp = requests.get(url, timeout=120)
        if resp.status_code == 200:
            zip_path = f'../data/raw/inmet/{year}.zip'
            with open(zip_path, 'wb') as f:
                f.write(resp.content)

            # Extrair
            with zipfile.ZipFile(zip_path, 'r') as z:
                z.extractall(f'../data/raw/inmet/{year}/')
            print(f"  ✅ {year}")
        else:
            print(f"  ❌ {year}: HTTP {resp.status_code}")
    except Exception as e:
        print(f"  ❌ {year}: {e}")


In [ ]:
def ler_csv_inmet(filepath):
    # Ler metadados (linhas 1-8)
    with open(filepath, 'r', encoding='latin-1') as f:
        meta = {}
        for i in range(8):
            line = f.readline().strip()
            if ':' in line:
                key, val = line.split(':', 1)
                meta[key.strip()] = val.strip().rstrip(';')

    # Ler dados a partir da linha 9
    df = pd.read_csv(
        filepath,
        sep=';',
        skiprows=8,
        encoding='latin-1',
        decimal=',',
        na_values=['-9999', '-9999.0', '']
    )

    # Adicionar metadados como colunas
    df['estacao_codigo'] = meta.get('CODIGO (WMO)', '')
    df['estacao_lat'] = float(meta.get('LATITUDE', '0').replace(',', '.'))
    df['estacao_lon'] = float(meta.get('LONGITUDE', '0').replace(',', '.'))
    df['estacao_alt'] = meta.get('ALTITUDE', '0')
    df['estacao_uf'] = meta.get('UF', '')
    df['estacao_nome'] = meta.get('ESTACAO', '')

    return df

# Processar todos os arquivos de um ano (exemplo 2022)
# all_files = glob.glob('../data/raw/inmet/2022/*.CSV')
# dfs = [ler_csv_inmet(f) for f in all_files]
# df_clima = pd.concat(dfs, ignore_index=True)
# print(f"Total de registros climáticos: {len(df_clima)}")


In [ ]:
# Via API SIDRA — Saneamento por município
# Tabela 3218: Domicílios por esgotamento sanitário
url_san = "https://apisidra.ibge.gov.br/values/t/3218/n6/all/v/all/p/last%201/c61/allxt"
resp = requests.get(url_san, timeout=300)
data_san = resp.json()
df_saneamento = pd.DataFrame(data_san[1:])
df_saneamento.to_csv('../data/raw/ibge/saneamento_municipios.csv', index=False)

# População Municipal
url_pop = "https://apisidra.ibge.gov.br/values/t/6579/n6/all/v/all/p/last%201"
resp = requests.get(url_pop)
data_pop = resp.json()
df_pop = pd.DataFrame(data_pop[1:])

import geobr
# municipios = geobr.read_municipality(year=2022)
